# Workshop 4 Live Demo: Connect Everything

## From Source Code to Result

In Workshop 3, we built:

```text
source code → tokens → AST
```

In this live demo, we connect the whole pipeline:

```text
source code → tokens → parser → AST → evaluator → result
```

## Demo Plan

1. Define tokens and AST nodes.
2. Use a tiny lexer.
3. Use a tiny parser.
4. Implement `evaluate()`.
5. Connect everything inside `run()`.
6. Show inspect mode.
7. Preview variables and environments.

In [ ]:
from dataclasses import dataclass
from enum import Enum, auto
from typing import Any

# Part 1 — Tokens and AST Nodes

These are the basic data structures from Workshop 3.

In [ ]:
class TokenType(Enum):
    NUMBER = auto()
    PLUS = auto()
    STAR = auto()
    EOF = auto()


@dataclass(frozen=True)
class Token:
    type: TokenType
    value: Any = None

    def __repr__(self):
        if self.value is None:
            return self.type.name
        return f"{self.type.name}({self.value})"

In [ ]:
@dataclass(frozen=True)
class Number:
    value: int


@dataclass(frozen=True)
class BinaryOp:
    op: str
    left: Any
    right: Any

# Part 2 — Lexer

The lexer turns source code text into tokens.

Lecture prompt:

> What should `tokenize("12 + 3")` produce?

In [ ]:
def tokenize(source: str) -> list[Token]:
    tokens = []
    i = 0

    while i < len(source):
        char = source[i]

        if char.isspace():
            i += 1
            continue

        if char.isdigit():
            start = i
            while i < len(source) and source[i].isdigit():
                i += 1
            tokens.append(Token(TokenType.NUMBER, int(source[start:i])))
            continue

        if char == "+":
            tokens.append(Token(TokenType.PLUS))
            i += 1
            continue

        if char == "*":
            tokens.append(Token(TokenType.STAR))
            i += 1
            continue

        raise SyntaxError(f"Unexpected character: {char!r}")

    tokens.append(Token(TokenType.EOF))
    return tokens

In [ ]:
tokens = tokenize("1 + 2 * 3")
tokens

# Part 3 — Parser

Grammar:

```text
expression → term (+ term)*
term       → factor (* factor)*
factor     → NUMBER
```

Because `*` lives deeper in the grammar, it binds more tightly.

In [ ]:
class Parser:
    def __init__(self, tokens: list[Token]):
        self.tokens = tokens
        self.current = 0

    def peek(self) -> Token:
        return self.tokens[self.current]

    def advance(self) -> Token:
        token = self.peek()
        self.current += 1
        return token

    def match(self, token_type: TokenType) -> bool:
        if self.peek().type == token_type:
            self.advance()
            return True
        return False

    def consume(self, token_type: TokenType, message: str) -> Token:
        if self.peek().type == token_type:
            return self.advance()
        raise SyntaxError(message)

    def parse(self):
        expr = self.parse_expression()
        self.consume(TokenType.EOF, "Expected end of expression")
        return expr

    def parse_expression(self):
        left = self.parse_term()

        while self.match(TokenType.PLUS):
            right = self.parse_term()
            left = BinaryOp("+", left, right)

        return left

    def parse_term(self):
        left = self.parse_factor()

        while self.match(TokenType.STAR):
            right = self.parse_factor()
            left = BinaryOp("*", left, right)

        return left

    def parse_factor(self):
        token = self.peek()

        if token.type == TokenType.NUMBER:
            self.advance()
            return Number(token.value)

        raise SyntaxError(f"Expected number, got {token}")

In [ ]:
ast = Parser(tokenize("1 + 2 * 3")).parse()
ast

## Pretty Print the AST

This helper makes the tree easier to discuss during lecture.

In [ ]:
def pretty_ast(node, indent: str = "") -> str:
    if isinstance(node, Number):
        return f"{indent}Number({node.value})"

    if isinstance(node, BinaryOp):
        left = pretty_ast(node.left, indent + "  ")
        right = pretty_ast(node.right, indent + "  ")

        return (
            f"{indent}BinaryOp(\n"
            f"{indent}  op={node.op!r},\n"
            f"{indent}  left=\n{left},\n"
            f"{indent}  right=\n{right}\n"
            f"{indent})"
        )

    raise TypeError(f"Unknown AST node: {node}")

In [ ]:
print(pretty_ast(ast))

# Part 4 — Live Coding: Evaluator

This is the main live-coding moment.

Ask students:

> What is the simplest AST node to evaluate?

Expected: `Number`

Then ask:

> What do we do for `BinaryOp`?

Expected: evaluate left, evaluate right, combine.

In [ ]:
# Starter version for live coding

def evaluate_live(node):
    # TODO 1:
    # If this is a Number, return its value.

    # TODO 2:
    # If this is a BinaryOp:
    #   evaluate the left child
    #   evaluate the right child
    #   combine based on the operator

    raise NotImplementedError("Live-code this function")

## Reference Solution

Use this after the live coding, or keep it hidden until students have reasoned through the algorithm.

In [ ]:
def evaluate(node):
    if isinstance(node, Number):
        return node.value

    if isinstance(node, BinaryOp):
        left = evaluate(node.left)
        right = evaluate(node.right)

        if node.op == "+":
            return left + right

        if node.op == "*":
            return left * right

        raise ValueError(f"Unknown operator: {node.op}")

    raise TypeError(f"Unknown AST node: {node}")

In [ ]:
evaluate(ast)

# Part 5 — Connect Everything

Now we combine all stages into one function.

Lecture prompt:

> Which function runs first?
> Which function runs last?

In [ ]:
def run(source: str):
    tokens = tokenize(source)
    ast = Parser(tokens).parse()
    result = evaluate(ast)
    return result

In [ ]:
run("1 + 2 * 3")

In [ ]:
examples = [
    "1 + 2",
    "2 * 3",
    "10 + 20 * 3",
    "1 + 2 + 3",
    "2 * 3 * 4",
]

for source in examples:
    print(source, "=>", run(source))

# Part 6 — Inspect Mode

Sometimes we don't just want the result.

We want to see every stage.

In [ ]:
def inspect(source: str):
    print("Source:")
    print(source)

    print("\nTokens:")
    tokens = tokenize(source)
    print(tokens)

    print("\nAST:")
    ast = Parser(tokens).parse()
    print(pretty_ast(ast))

    print("\nResult:")
    print(evaluate(ast))

In [ ]:
inspect("1 + 2 * 3")

# Part 7 — Mini REPL-Style Demo

This simulates a few REPL commands without using `input()`.

In [ ]:
commands = [
    "1 + 2",
    "1 + 2 * 3",
    "10 * 2 + 5",
]

for command in commands:
    print(">>>", command)
    print(run(command))
    print()

# Part 8 — Preview: Why Variables Need an Environment

Right now, every run is independent.

Next, we want:

```text
>>> x = 5
>>> x + 2
7
```

That requires an environment:

```python
env = {"x": 5}
```

In [ ]:
env = {}

# Preview only — not part of today's completed evaluator.
env["x"] = 5
env["x"] + 2

# Lecture Closing

Students should now understand:

```text
Source Code
↓
Lexer
↓
Tokens
↓
Parser
↓
AST
↓
Evaluator
↓
Result
```

Workshop 4 extends this with:

```text
Environment
Variables
State
```